# 20: MILK10k Silver validation

Normalizes labels, validates images, assigns leakage groups and records rejections, using the shared code in `data_platform.spark_io` with MILK10k's settings from `config/bronze/datasets/milk10k.yaml`. Run after `10_bronze_ingest`.

Produces this dataset's rows in `silver.image_inventory`, `silver.leakage_groups` and `silver.rejected_records`.

In [ ]:
%run ../_setup_env


In [ ]:
from pathlib import Path
import tempfile

from data_platform.files import check_archives_exist, stage_archives_and_extract_metadata
from data_platform.labels import MALIGNANCY_VALUES, normalize_diagnosis_labels
from data_platform.dataset_layout import load_dataset
from data_platform.spark_io import (
    Rejections,
    apply_label_normalization,
    assign_leakage_groups,
    build_accepted_rows,
    print_silver_outputs,
    reconcile_bronze_records,
    scratch_table_name,
    validate_images,
    write_silver_dataset,
)

dataset = load_dataset(CONFIG_ROOT, "milk10k")
DATASET_CONFIG = dataset["config"]
ARCHIVES = dataset["archives"]
bronze_tables, silver_tables = dataset["bronze_tables"], dataset["silver_tables"]

# Everything dataset-specific comes from config/bronze/datasets/milk10k.yaml.
DATASET_KEY = DATASET_CONFIG["dataset_key"]
DIAGNOSIS_INPUT_COLUMNS = DATASET_CONFIG["diagnosis_input_columns"]
LABEL_COLUMNS = DATASET_CONFIG["label_columns"]
LOCAL_ARCHIVE_STAGE_ROOT = Path(tempfile.gettempdir()) / DATASET_CONFIG["dataset_key"] / "archive_stage"

print(f"Bronze source metadata table: {bronze_tables['source_metadata']}")
print(f"Bronze image index table: {bronze_tables['image_index']}")
print(f"Silver image inventory table: {silver_tables['image_inventory']}")
print(f"Silver leakage groups table: {silver_tables['leakage_groups']}")
print(f"Silver rejected records table: {silver_tables['rejected_records']}")


## Stage archives locally

Silver reads images straight from the archives, so it needs its own local copy.

In [ ]:
check_archives_exist(ARCHIVES)
stage_archives_and_extract_metadata(ARCHIVES, LOCAL_ARCHIVE_STAGE_ROOT)


## Reject images with no metadata

In [ ]:
# Collects this run's rejections; they're written together with the accepted rows at the end.
rejections = Rejections(DATASET_KEY)
source_metadata_df = reconcile_bronze_records(spark, bronze_tables, rejections)


## Normalize labels

Maps the diagnosis columns to `malignancy` and `specific_diagnosis`. A `malignancy` value outside `benign`, `malignant` and `indeterminate` fails the run.

In [ ]:
label_valid_df = apply_label_normalization(
    source_metadata_df,
    normalize_diagnosis_labels,
    DIAGNOSIS_INPUT_COLUMNS,
    LABEL_COLUMNS,
    rejections,
    controlled_vocabularies={"malignancy": MALIGNANCY_VALUES},
    scratch_table=scratch_table_name(silver_tables, DATASET_KEY, "labeled_rows"),
)


## Validate images

Streams each candidate from its archive, re-checks its checksum, and decodes it. A checksum mismatch fails the whole run ([ADR 005](../../docs/decisions/005-immutable-source-archives-checksum-verified.md)).

In [ ]:
image_valid_df = validate_images(spark, label_valid_df, ARCHIVES, rejections)


## Assign leakage groups and write Silver

Groups by identical image, then lesion, then patient, within this dataset only. Then replaces this dataset's rows in all three Silver tables, so a rerun never leaves rows from an earlier run behind.

In [ ]:
accepted_df = build_accepted_rows(label_valid_df, image_valid_df, LABEL_COLUMNS, DATASET_KEY)
image_inventory_df, leakage_groups_df = assign_leakage_groups(accepted_df, LABEL_COLUMNS)
write_silver_dataset(spark, silver_tables, image_inventory_df, leakage_groups_df, rejections)


## Review

In [ ]:
print_silver_outputs(spark, display, silver_tables, LABEL_COLUMNS, DATASET_KEY)
